In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F, Window
from pyspark.sql.types import StringType, ArrayType
from datetime import datetime

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

catalog = "cdac-project"

# One schema per document type — matches Notebook 4, which writes
# each type's structured output into its own schema.
DOCUMENT_TYPE_SCHEMAS = {
    "RESUME": "resume",
    "EMAIL": "email",
    "INVOICE": "invoice",
    "BANK_STATEMENT": "bank_statement",
    "PRESCRIPTION": "prescription",
}

for schema_name in DOCUMENT_TYPE_SCHEMAS.values():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema_name}`")

# Input: Notebook 4's per-type structured output.
resume_structured_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['RESUME']}`.resume_structured_data"
email_structured_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['EMAIL']}`.email_structured_data"
invoice_structured_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['INVOICE']}`.invoice_structured_data"
bank_statement_structured_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['BANK_STATEMENT']}`.bank_statement_structured_data"
prescription_structured_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['PRESCRIPTION']}`.prescription_structured_data"

# Output: this notebook's validated result, one table per type.
validated_resume_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['RESUME']}`.validated_resume_data"
validated_email_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['EMAIL']}`.validated_email_data"
validated_invoice_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['INVOICE']}`.validated_invoice_data"
validated_bank_statement_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['BANK_STATEMENT']}`.validated_bank_statement_data"
validated_prescription_table = f"`{catalog}`.`{DOCUMENT_TYPE_SCHEMAS['PRESCRIPTION']}`.validated_prescription_data"

EMAIL_PATTERN = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
AMOUNT_PATTERN = r"^\$?[\d,]+(?:\.\d{1,2})?$"

In [ ]:
# ============================================================
# 3. STRUCTURED TABLE FALLBACK SCHEMAS
# ============================================================
# Used when a notebook run happens before Notebook 4 has ever
# produced a row for that document type — spark.table() on a
# missing table raises an unhandled AnalysisException otherwise.
# Each schema matches the corresponding OUTPUT_CONFIG DDL in
# Notebook 4 exactly, plus processing_timestamp.

RESUME_STRUCTURED_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, name STRING, email STRING, phone STRING,
linkedin STRING, github STRING, skills ARRAY<STRING>, education ARRAY<STRING>,
experience ARRAY<STRING>, projects ARRAY<STRING>, certifications ARRAY<STRING>,
summary STRING, processing_status STRING, error_message STRING,
processing_timestamp TIMESTAMP
"""

EMAIL_STRUCTURED_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, sender STRING, recipient STRING,
subject STRING, sent_date STRING, body STRING, processing_status STRING,
error_message STRING, processing_timestamp TIMESTAMP
"""

INVOICE_STRUCTURED_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, invoice_number STRING, invoice_date STRING,
due_date STRING, total_amount STRING, processing_status STRING,
error_message STRING, processing_timestamp TIMESTAMP
"""

BANK_STATEMENT_STRUCTURED_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, account_number STRING, statement_period STRING,
opening_balance STRING, closing_balance STRING, transactions ARRAY<STRING>,
processing_status STRING, error_message STRING, processing_timestamp TIMESTAMP
"""

PRESCRIPTION_STRUCTURED_FALLBACK_SCHEMA = """
file_id STRING, document_type STRING, patient_name STRING, physician_name STRING,
prescription_date STRING, diagnosis STRING, medications ARRAY<STRING>,
processing_status STRING, error_message STRING, processing_timestamp TIMESTAMP
"""

In [ ]:
# ============================================================
# 4. VALIDATION FUNCTIONS
# ============================================================
# Each function takes a DataFrame and returns a DataFrame with new
# columns added — small, single-purpose, and composable. All of them
# are parametrized (which columns to check) rather than hardcoded to
# resume's fields, so the same functions serve every document type;
# section 5 (VALIDATION_CONFIG) supplies the per-type parameters and
# section 6 wires them together. None of these collect() or touch
# the driver; they all compile down to ordinary Spark column
# expressions, so this scales the same regardless of table size.

def validate_data_types(df, list_columns):
    """
    Ensures list-type fields are genuine arrays, not a single
    delimited string. Notebook 4 already stores these as
    ARRAY<STRING>, so in this pipeline this is mostly a defensive
    check for a hand-edited table where a list field might arrive as
    e.g. "Amoxicillin, Paracetamol" instead of ["Amoxicillin",
    "Paracetamol"] — inspecting df.schema is a metadata-only
    operation, not a data scan, so this stays cheap even on a huge
    table.
    """

    for column in list_columns:
        if column not in df.columns:
            continue

        if isinstance(df.schema[column].dataType, StringType):
            df = df.withColumn(
                column,
                F.when(
                    F.col(column).isNull() | (F.trim(F.col(column)) == ""),
                    F.array().cast("array<string>")
                ).otherwise(F.split(F.col(column), r"\s*,\s*"))
            )

    return df


def validate_required_fields(df, required_fields):
    """
    Adds `missing_required_fields`: an array naming every mandatory
    field that's null/blank (for a string field) or empty (for an
    array field, e.g. a prescription with zero medications) on that
    row. An empty array means nothing required is missing.
    """

    missing_checks = []
    for f in required_fields:
        if isinstance(df.schema[f].dataType, ArrayType):
            condition = F.col(f).isNull() | (F.size(F.col(f)) == 0)
        else:
            condition = F.col(f).isNull() | (F.trim(F.col(f)) == "")
        missing_checks.append(F.when(condition, F.lit(f)))

    return df.withColumn(
        "missing_required_fields",
        F.filter(F.array(*missing_checks), lambda x: x.isNotNull())
    )


def validate_email(df, column):
    """
    Adds `<column>_validation_status`: PASS if the address has a
    proper local-part@domain.extension shape (this is what rules out
    something like "john@gmail" — no dot, no extension), FAILED
    otherwise, or None if the field itself is empty (missing is a
    separate concern, handled by validate_required_fields). Used for
    resume's "email" and email's "sender"/"recipient" — same shape
    check either way.
    """

    status_column = f"{column}_validation_status"

    return df.withColumn(
        status_column,
        F.when(F.col(column).isNull() | (F.trim(F.col(column)) == ""), F.lit(None))
         .when(F.col(column).rlike(EMAIL_PATTERN), F.lit("PASS"))
         .otherwise(F.lit("FAILED"))
    )


def validate_phone(df, column):
    """
    Strips everything but digits, then strips a leading "91" country
    code if what's left is 12 digits (e.g. "+91 98765 43210" ->
    "9876543210") — adds `phone_normalized` (the cleaned number) and
    `<column>_validation_status` (PASS if the normalized number is a
    plausible 7-15 digit length, FAILED otherwise, None if empty).
    Resume-only — no other document type has a phone field.
    """

    status_column = f"{column}_validation_status"
    digits = F.regexp_replace(F.coalesce(F.col(column), F.lit("")), r"[^0-9]", "")

    df = df.withColumn("_phone_digits", digits)
    df = df.withColumn(
        "phone_normalized",
        F.when(
            (F.length(F.col("_phone_digits")) == 12) & F.col("_phone_digits").startswith("91"),
            F.expr("substring(_phone_digits, 3, 10)")
        ).otherwise(F.col("_phone_digits"))
    )
    df = df.withColumn(
        status_column,
        F.when(F.col(column).isNull() | (F.trim(F.col(column)) == ""), F.lit(None))
         .when(
             (F.length(F.col("phone_normalized")) >= 7) & (F.length(F.col("phone_normalized")) <= 15),
             F.lit("PASS")
         )
         .otherwise(F.lit("FAILED"))
    )
    df = df.drop("_phone_digits")

    return df


def validate_url(df, column):
    """
    Adds `<column>_validation_status`: PASS if the URL starts with
    http:// or https://, FAILED otherwise, None if empty. Used for
    resume's linkedin and github.
    """

    status_column = f"{column}_validation_status"

    return df.withColumn(
        status_column,
        F.when(F.col(column).isNull() | (F.trim(F.col(column)) == ""), F.lit(None))
         .when(F.col(column).rlike(r"^https?://"), F.lit("PASS"))
         .otherwise(F.lit("FAILED"))
    )


def validate_amount(df, column):
    """
    Adds `<column>_validation_status`: PASS if the value looks like
    a real money amount (optional "$", digits with optional comma
    grouping, optional 1-2 decimal places), FAILED otherwise, None
    if empty. Used for invoice's total_amount and bank statement's
    opening/closing balance.
    """

    status_column = f"{column}_validation_status"

    return df.withColumn(
        status_column,
        F.when(F.col(column).isNull() | (F.trim(F.col(column)) == ""), F.lit(None))
         .when(F.col(column).rlike(AMOUNT_PATTERN), F.lit("PASS"))
         .otherwise(F.lit("FAILED"))
    )


def check_duplicate_records(df, identity_columns):
    """
    Adds `duplicate_flag`: True if this row shares its file_id with
    another row (shouldn't normally happen — file_id comes from a
    hash of the file's path+size back in Notebook 1 — but the spec
    calls it out explicitly as a check), OR shares the same
    combination of `identity_columns` (case-insensitive, trimmed)
    with another row. What counts as a duplicate differs by document
    type — two resumes are the same person if email+name match, two
    invoices are the same invoice if invoice_number matches, etc. —
    so VALIDATION_CONFIG supplies the right columns per type. Rows
    missing any identity column are never identity-matched against
    each other, so two blank records don't get flagged as duplicates
    of one another. Implemented with window functions, not a
    self-join or collect() — the standard way to do this at scale in
    Spark.
    """

    file_id_window = Window.partitionBy("file_id")
    df = df.withColumn("_file_id_count", F.count("*").over(file_id_window))

    if identity_columns:
        identity_exprs = [F.lower(F.trim(F.col(c))) for c in identity_columns]
        identity_window = Window.partitionBy(*identity_exprs)

        all_identity_present = F.lit(True)
        for c in identity_columns:
            all_identity_present = all_identity_present & F.col(c).isNotNull() & (F.trim(F.col(c)) != "")

        df = df.withColumn(
            "_identity_count",
            F.when(all_identity_present, F.count("*").over(identity_window)).otherwise(F.lit(1))
        )
    else:
        df = df.withColumn("_identity_count", F.lit(1))

    df = df.withColumn("duplicate_flag", (F.col("_file_id_count") > 1) | (F.col("_identity_count") > 1))
    df = df.drop("_file_id_count", "_identity_count")

    return df


def calculate_confidence_score(df, field_weights, list_score_fields):
    """
    Adds `extraction_confidence_score` (0-100): points awarded per
    field actually present, using the weights VALIDATION_CONFIG
    assigns for that document type (they add up to 100 per type).
    Fields in `list_score_fields` are scored by F.size(...) > 0
    (e.g. "at least one medication was extracted") rather than a
    blank-string check. This measures how COMPLETE the extraction
    was, not whether the values present are correct — that's what
    validation_status covers.
    """

    score = F.lit(0)
    for field_name, points in field_weights.items():
        if field_name in list_score_fields:
            condition = F.size(F.col(field_name)) > 0
        else:
            condition = F.col(field_name).isNotNull() & (F.trim(F.col(field_name)) != "")
        score = score + F.when(condition, F.lit(points)).otherwise(F.lit(0))

    return df.withColumn("extraction_confidence_score", score)


def build_validation_summary(df, status_columns, hard_fail_columns):
    """
    Combines the individual per-field checks (already-added columns
    from the functions above) into one overall `validation_status`
    and a human-readable `validation_errors` list.

    `status_columns` is every "<field>_validation_status" column
    this document type has, each paired with the human-readable
    label to use if it's FAILED. `hard_fail_columns` is the subset
    of those column names that should fail the WHOLE record (in
    addition to any missing required field) — e.g. resume's email
    must be valid, invoice's total_amount must be valid, but
    resume's phone/linkedin/github being malformed still leaves a
    usable record, so those stay soft (recorded in validation_errors,
    but don't flip validation_status to FAILED).
    """

    error_exprs = [
        F.when(
            F.size(F.col("missing_required_fields")) > 0,
            F.concat(F.lit("Missing "), F.array_join(F.col("missing_required_fields"), ", "))
        )
    ]
    for status_column, label in status_columns:
        error_exprs.append(F.when(F.col(status_column) == "FAILED", F.lit(label)))

    error_parts = F.filter(F.array(*error_exprs), lambda x: x.isNotNull())

    df = df.withColumn(
        "validation_errors",
        F.when(F.size(error_parts) == 0, F.lit(None)).otherwise(error_parts)
    )

    fail_condition = F.size(F.col("missing_required_fields")) > 0
    for status_column in hard_fail_columns:
        fail_condition = fail_condition | (F.col(status_column) == "FAILED")

    df = df.withColumn(
        "validation_status",
        F.when(fail_condition, F.lit("FAILED")).otherwise(F.lit("PASS"))
    )

    return df

In [ ]:
# ============================================================
# 5. VALIDATION CONFIG
# ============================================================
# One entry per document type — this is what turns the generic
# functions in section 4 into resume-specific / email-specific /
# etc. behavior. Adding a 6th document type later means adding one
# entry here (plus its fallback schema in section 3) — section 6's
# pipeline loop stays unchanged.

VALIDATION_CONFIG = {
    "RESUME": {
        "structured_table": resume_structured_table,
        "validated_table": validated_resume_table,
        "fallback_schema": RESUME_STRUCTURED_FALLBACK_SCHEMA,
        "required_fields": ["name", "email"],
        "list_columns": ["skills", "education", "experience", "projects", "certifications"],
        "email_fields": ["email"],
        "url_fields": ["linkedin", "github"],
        "amount_fields": [],
        "phone_field": "phone",
        "identity_columns": ["email", "name"],
        "confidence_weights": {"name": 20, "email": 20, "phone": 10, "skills": 20, "education": 15, "experience": 15},
        "list_score_fields": {"skills", "education", "experience"},
        "hard_fail_columns": ["email_validation_status"],
    },
    "EMAIL": {
        "structured_table": email_structured_table,
        "validated_table": validated_email_table,
        "fallback_schema": EMAIL_STRUCTURED_FALLBACK_SCHEMA,
        "required_fields": ["sender", "subject"],
        "list_columns": [],
        "email_fields": ["sender", "recipient"],
        "url_fields": [],
        "amount_fields": [],
        "phone_field": None,
        "identity_columns": ["sender", "subject", "sent_date"],
        "confidence_weights": {"sender": 25, "recipient": 15, "subject": 25, "sent_date": 10, "body": 25},
        "list_score_fields": set(),
        "hard_fail_columns": ["sender_validation_status"],
    },
    "INVOICE": {
        "structured_table": invoice_structured_table,
        "validated_table": validated_invoice_table,
        "fallback_schema": INVOICE_STRUCTURED_FALLBACK_SCHEMA,
        "required_fields": ["invoice_number", "total_amount"],
        "list_columns": [],
        "email_fields": [],
        "url_fields": [],
        "amount_fields": ["total_amount"],
        "phone_field": None,
        "identity_columns": ["invoice_number"],
        "confidence_weights": {"invoice_number": 30, "invoice_date": 15, "due_date": 15, "total_amount": 40},
        "list_score_fields": set(),
        "hard_fail_columns": ["total_amount_validation_status"],
    },
    "BANK_STATEMENT": {
        "structured_table": bank_statement_structured_table,
        "validated_table": validated_bank_statement_table,
        "fallback_schema": BANK_STATEMENT_STRUCTURED_FALLBACK_SCHEMA,
        "required_fields": ["account_number", "closing_balance"],
        "list_columns": ["transactions"],
        "email_fields": [],
        "url_fields": [],
        "amount_fields": ["opening_balance", "closing_balance"],
        "phone_field": None,
        "identity_columns": ["account_number", "statement_period"],
        "confidence_weights": {"account_number": 25, "statement_period": 15, "opening_balance": 15, "closing_balance": 20, "transactions": 25},
        "list_score_fields": {"transactions"},
        "hard_fail_columns": ["closing_balance_validation_status"],
    },
    "PRESCRIPTION": {
        "structured_table": prescription_structured_table,
        "validated_table": validated_prescription_table,
        "fallback_schema": PRESCRIPTION_STRUCTURED_FALLBACK_SCHEMA,
        "required_fields": ["patient_name", "medications"],
        "list_columns": ["medications"],
        "email_fields": [],
        "url_fields": [],
        "amount_fields": [],
        "phone_field": None,
        "identity_columns": ["patient_name", "prescription_date", "physician_name"],
        "confidence_weights": {"patient_name": 25, "physician_name": 15, "prescription_date": 10, "diagnosis": 20, "medications": 30},
        "list_score_fields": {"medications"},
        "hard_fail_columns": [],
    },
}

In [ ]:
# ============================================================
# 6. RUN VALIDATION PIPELINE FOR EVERY DOCUMENT TYPE
# ============================================================
# validate_document_type() runs the full pipeline — read (with
# fallback), apply every validator VALIDATION_CONFIG lists for this
# type, duplicate check, confidence score, summary, then an
# incremental write — for ONE document type. The loop at the bottom
# runs it once per type. Same shape as Notebook 4's registry pattern:
# the logic lives in one place, and VALIDATION_CONFIG is what makes
# it behave differently per document type.
#
# This still reads/validates the FULL structured table every run,
# not just new rows — intentional. Duplicate detection needs to
# compare each record against ALL other records of that type to
# catch duplicates correctly, including one uploaded in a different
# pipeline run. Only the WRITE is filtered down to new file_ids, so
# a run with no new documents of a given type writes zero rows for it.

def validate_document_type(document_type, config):
    structured_table = config["structured_table"]
    validated_table = config["validated_table"]

    if spark.catalog.tableExists(structured_table):
        df = spark.table(structured_table)
    else:
        df = spark.createDataFrame([], schema=config["fallback_schema"])

    df = validate_data_types(df, config["list_columns"])
    df = validate_required_fields(df, config["required_fields"])

    status_columns = []

    for column in config["email_fields"]:
        df = validate_email(df, column)
        status_columns.append((f"{column}_validation_status", f"Invalid {column} format"))

    for column in config["url_fields"]:
        df = validate_url(df, column)
        status_columns.append((f"{column}_validation_status", f"Invalid {column} URL"))

    for column in config["amount_fields"]:
        df = validate_amount(df, column)
        status_columns.append((f"{column}_validation_status", f"Invalid {column} format"))

    if config["phone_field"]:
        column = config["phone_field"]
        df = validate_phone(df, column)
        status_columns.append((f"{column}_validation_status", f"Invalid {column} number"))

    df = build_validation_summary(df, status_columns, config["hard_fail_columns"])
    df = check_duplicate_records(df, config["identity_columns"])
    df = calculate_confidence_score(df, config["confidence_weights"], config["list_score_fields"])
    df = df.withColumn("validation_timestamp", F.current_timestamp())

    # Incremental — only keep rows whose file_id isn't already in
    # this type's validated table, so re-running this notebook with
    # no new documents of that type writes nothing (a true no-op)
    # instead of re-validating and re-writing every record every time.
    if spark.catalog.tableExists(validated_table):
        already_validated_ids = {
            row.file_id
            for row in spark.table(validated_table).select("file_id").collect()
        }
    else:
        already_validated_ids = set()

    df_to_write = (
        df.filter(~F.col("file_id").isin(already_validated_ids))
        if already_validated_ids
        else df
    )

    df_to_write.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(validated_table)


for document_type, config in VALIDATION_CONFIG.items():
    validate_document_type(document_type, config)
    print(f"Validated {document_type}")

In [ ]:
# ============================================================
# 7. DISPLAY VALIDATION RESULTS
# ============================================================

for document_type, config in VALIDATION_CONFIG.items():
    validated_table = config["validated_table"]

    if not spark.catalog.tableExists(validated_table):
        continue

    print(f"--- {document_type} ---")
    display(spark.table(validated_table))

    summary_df = (
        spark.table(validated_table)
        .groupBy("validation_status")
        .agg(
            F.count("*").alias("record_count"),
            F.round(F.avg("extraction_confidence_score"), 1).alias("avg_confidence_score"),
            F.sum(F.col("duplicate_flag").cast("int")).alias("duplicate_count"),
        )
    )
    display(summary_df)